In [ ]:
# 1) Cargar la base de datos (sube el archivo Datos_históricos_RKLB.csv)
from google.colab import files
import io, numpy as np, pandas as pd, matplotlib.pyplot as plt

subido = files.upload()
nombre = list(subido.keys())[0]
df = pd.read_csv(io.BytesIO(subido[nombre]))

# Limpieza: fechas dd.mm.yyyy, números y volumen ("23.31M")
df["Fecha"] = pd.to_datetime(df["Fecha"], format="%d.%m.%Y")
df["Vol."] = df["Vol."].str.replace("M", "", regex=False).astype(float) * 1e6
df["% var."] = df["% var."].str.replace("%", "", regex=False).astype(float)
df = df.sort_values("Fecha").reset_index(drop=True)

print(f"{len(df)} meses: {df['Fecha'].min().date()} a {df['Fecha'].max().date()}")
df.tail()

In [ ]:
# 2) Serie mensual y transformación logarítmica
serie = df.set_index("Fecha")["Cierre"]
serie.index = serie.index.to_period("M").to_timestamp()
serie = serie.asfreq("MS")
y_log = np.log(serie)

# Prueba de estacionariedad (ADF): p < 0.05 => estacionaria
from statsmodels.tsa.stattools import adfuller
for nombre, s in [("log(precio)", y_log), ("Δ log(precio) (rendimiento)", y_log.diff().dropna())]:
    p = adfuller(s)[1]
    print(f"ADF {nombre}: p-valor = {p:.4f} -> {'estacionaria' if p < 0.05 else 'NO estacionaria'}")

In [ ]:
# 3) Selección del modelo ARIMA(p,d,q) por AIC
import itertools, warnings
from statsmodels.tsa.arima.model import ARIMA
warnings.filterwarnings("ignore")

resultados = []
for p, d, q in itertools.product(range(0, 4), [1], range(0, 4)):
    try:
        mod = ARIMA(y_log, order=(p, d, q), trend="t").fit()   # trend="t": deriva (drift)
        resultados.append((p, d, q, mod.aic))
    except Exception:
        pass

tabla_aic = pd.DataFrame(resultados, columns=["p", "d", "q", "AIC"]).sort_values("AIC").reset_index(drop=True)
display(tabla_aic.head(5).round(2))

p, d, q = tabla_aic.loc[0, ["p", "d", "q"]].astype(int)
modelo = ARIMA(y_log, order=(p, d, q), trend="t").fit()
print(f"\nModelo elegido: ARIMA({p},{d},{q}) con deriva")
print(modelo.summary().tables[1])

In [ ]:
# 4) Diagnóstico de residuales
from statsmodels.stats.diagnostic import acorr_ljungbox
lb = acorr_ljungbox(modelo.resid[1:], lags=[12], return_df=True)
print(f"Ljung-Box (12 rezagos): p-valor = {lb['lb_pvalue'].iloc[0]:.4f} "
      f"-> {'residuales sin autocorrelación (bien)' if lb['lb_pvalue'].iloc[0] > 0.05 else 'queda autocorrelación'}")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(modelo.resid[1:]); ax[0].set_title("Residuales"); ax[0].grid(alpha=0.3)
ax[1].hist(modelo.resid[1:], bins=15, color="tab:blue", edgecolor="white"); ax[1].set_title("Histograma de residuales")
plt.show()

In [ ]:
# 5) Pronóstico ARIMA a 12 meses (nov-2026 ... oct-2027)
h = 12
pron = modelo.get_forecast(steps=h)
media = np.exp(pron.predicted_mean)
ic80 = np.exp(pron.conf_int(alpha=0.20))
ic95 = np.exp(pron.conf_int(alpha=0.05))

res = pd.DataFrame({
    "Pronóstico": media,
    "Inf 80%": ic80.iloc[:, 0], "Sup 80%": ic80.iloc[:, 1],
    "Inf 95%": ic95.iloc[:, 0], "Sup 95%": ic95.iloc[:, 1]
}).round(2)
res.index = res.index.strftime("%b-%Y")
display(res)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(serie, color="tab:blue", linewidth=2.5, label="Histórico")
ax.plot(media, color="tab:red", linewidth=2.5, label=f"ARIMA({p},{d},{q}) pronóstico")
ax.fill_between(media.index, ic95.iloc[:, 0], ic95.iloc[:, 1], color="tab:red", alpha=0.10, label="IC 95%")
ax.fill_between(media.index, ic80.iloc[:, 0], ic80.iloc[:, 1], color="tab:red", alpha=0.20, label="IC 80%")
ax.axvline(serie.index[-1], color="gray", linestyle=":")
ax.set_yscale("log")
ax.set_title("RKLB: pronóstico ARIMA del cierre mensual hasta octubre 2027", fontsize=15, fontweight="bold")
ax.set_ylabel("Precio de cierre (USD, escala log)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

print(f"Último cierre ({serie.index[-1].strftime('%b-%Y')}): {serie.iloc[-1]:.2f}")
print(f"Octubre 2027 -> ARIMA: {media.iloc[-1]:.2f} "
      f"(IC 80%: {ic80.iloc[-1, 0]:.2f} a {ic80.iloc[-1, 1]:.2f}; IC 95%: {ic95.iloc[-1, 0]:.2f} a {ic95.iloc[-1, 1]:.2f})")